<a href="https://colab.research.google.com/github/Fahad-Hafeez/safecalib/blob/main/04_bootstrap_ci.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

FUNCTION bootstrap_metric(df, metric_fn, n_iterations=1000, ci=0.95):
    bootstrap_values = []

    FOR _ in range(n_iterations):
        # Resample with replacement
        sample = df.sample(n=len(df), replace=True, random_state=None)
        value = metric_fn(sample)
        bootstrap_values.append(value)

    lower = np.percentile(bootstrap_values, (1 - ci) / 2 * 100)
    upper = np.percentile(bootstrap_values, (1 + ci) / 2 * 100)
    mean  = np.mean(bootstrap_values)

    RETURN mean, lower, upper

In [ ]:
ci_results = {}

FOR model in all_models:
    model_df = clean_df[clean_df['model'] == model]

    ci_results[model] = {
        'URR_L1':    bootstrap_metric(model_df, lambda d: compute_urr_at_level(d, 1)),
        'URR_L5':    bootstrap_metric(model_df, lambda d: compute_urr_at_level(d, 5)),
        'ORR':       bootstrap_metric(model_df, lambda d: compute_orr(d)),
        'CA_ECE':    bootstrap_metric(model_df, lambda d: compute_ca_ece(d)),
        'ACS':       bootstrap_metric(model_df, lambda d: compute_acs(d)),
    }

    # Format for paper table: "mean (95% CI: lower–upper)"
    FOR metric, (mean, lo, hi) in ci_results[model].items():
        print(f"{model} | {metric}: {mean:.3f} ({lo:.3f}–{hi:.3f})")

In [ ]:
FUNCTION format_to_latex(ci_results):
    # Outputs ready-to-paste LaTeX table rows
    FOR model in instruct_models:
        row_parts = [model.replace('_', '-')]
        FOR metric in ['URR_L1', 'URR_L5', 'ORR', 'CA_ECE', 'ACS']:
            mean, lo, hi = ci_results[model][metric]
            row_parts.append(f"{mean:.3f}\\textsubscript{{({lo:.3f}–{hi:.3f})}}")
        print(" & ".join(row_parts) + " \\\\")